# 05 — ML end-to-end aplicado a ATM

## Proyecto final del bloque de Machine Learning fundamental

Este notebook integra en un único flujo los conceptos vistos hasta ahora:

- variables numéricas y categóricas;
- preparación de datos;
- clasificación;
- baseline;
- `train / validation / test`;
- pipelines;
- cross-validation;
- hiperparámetros;
- overfitting;
- comparación de modelos;
- métricas de clasificación;
- selección de modelo;
- evaluación final sobre **test no utilizado durante el desarrollo**;
- interpretación básica del modelo;
- uso del modelo sobre una situación ATM nueva.

El objetivo no es construir un modelo ATM operacional real, sino recorrer correctamente el **ciclo completo de un proyecto de ML supervisado**.

---

## Caso de uso

Vamos a simular registros operacionales de sectores de espacio aéreo.

Queremos predecir:

> **¿Existe riesgo de una situación de alta carga operacional que requiera una medida de gestión de capacidad?**

La variable objetivo será:

`capacity_event`

- `0`: situación normal.
- `1`: situación de alta carga/capacidad.

Los datos serán sintéticos para poder concentrarnos en el proceso de ML y no en la adquisición o limpieza de datos reales.

## 1. Importaciones

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    GridSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

pd.set_option("display.max_columns", 50)

## 2. Generación de un dataset ATM sintético

Cada fila representa una situación operacional de un sector.

### Variables numéricas

- `traffic_count`: aeronaves previstas en la ventana temporal.
- `sector_capacity`: capacidad nominal del sector.
- `weather_severity`: severidad meteorológica, de 0 a 10.
- `avg_delay_min`: demora media existente.
- `staffing_ratio`: proporción de dotación disponible.
- `coordination_load`: carga de coordinación, de 0 a 10.

### Variables categóricas

- `sector_type`: tipo de sector.
- `time_band`: franja horaria.
- `flow_state`: estado general del flujo.

### Target

- `capacity_event`: 1 cuando la combinación de factores genera una situación de alta carga operacional.

La función que genera el target incluye ruido y algunas relaciones no lineales. Por tanto, ningún modelo debería ser perfecto.

In [ ]:
n = 5000

traffic_count = rng.integers(10, 55, n)
sector_capacity = rng.integers(24, 46, n)
weather_severity = rng.uniform(0, 10, n)
avg_delay_min = np.clip(rng.normal(8, 7, n), 0, None)
staffing_ratio = np.clip(rng.normal(0.94, 0.08, n), 0.65, 1.10)
coordination_load = rng.uniform(0, 10, n)

sector_type = rng.choice(
    ["ACC", "TMA", "APP"],
    size=n,
    p=[0.50, 0.30, 0.20]
)

time_band = rng.choice(
    ["morning", "afternoon", "evening", "night"],
    size=n,
    p=[0.30, 0.35, 0.25, 0.10]
)

flow_state = rng.choice(
    ["stable", "growing", "volatile"],
    size=n,
    p=[0.55, 0.30, 0.15]
)

traffic_ratio = traffic_count / sector_capacity

sector_effect = np.select(
    [
        sector_type == "ACC",
        sector_type == "TMA",
        sector_type == "APP",
    ],
    [0.35, 0.65, 0.45]
)

time_effect = np.select(
    [
        time_band == "morning",
        time_band == "afternoon",
        time_band == "evening",
        time_band == "night",
    ],
    [0.25, 0.55, 0.35, -0.25]
)

flow_effect = np.select(
    [
        flow_state == "stable",
        flow_state == "growing",
        flow_state == "volatile",
    ],
    [-0.45, 0.35, 0.95]
)

# Señal operacional sintética.
risk_score = (
    3.2 * (traffic_ratio - 0.85)
    + 0.20 * weather_severity
    + 0.045 * avg_delay_min
    + 0.17 * coordination_load
    - 2.2 * (staffing_ratio - 0.90)
    + sector_effect
    + time_effect
    + flow_effect
    + 1.4 * ((traffic_ratio > 1.05) & (weather_severity > 6))
    + rng.normal(0, 1.15, n)
)

capacity_event = (risk_score > 2.65).astype(int)

df = pd.DataFrame({
    "traffic_count": traffic_count,
    "sector_capacity": sector_capacity,
    "weather_severity": weather_severity,
    "avg_delay_min": avg_delay_min,
    "staffing_ratio": staffing_ratio,
    "coordination_load": coordination_load,
    "sector_type": sector_type,
    "time_band": time_band,
    "flow_state": flow_state,
    "capacity_event": capacity_event,
})

df.head()

## 3. Exploración inicial

Antes de modelar, comprobamos:

1. tamaño del dataset;
2. tipos de variables;
3. valores ausentes;
4. distribución de la variable objetivo;
5. algunas relaciones simples.

En un proyecto real esta etapa sería mucho más extensa.

In [ ]:
print("Shape:", df.shape)
print()
df.info()

In [ ]:
df.isna().sum()

In [ ]:
target_distribution = (
    df["capacity_event"]
    .value_counts(normalize=True)
    .rename(index={0: "normal", 1: "capacity_event"})
)

target_distribution

In [ ]:
df.groupby("flow_state")["capacity_event"].mean().sort_values(ascending=False)

In [ ]:
df.groupby("sector_type")["capacity_event"].mean().sort_values(ascending=False)

In [ ]:
plt.hist(
    [
        df.loc[df["capacity_event"] == 0, "weather_severity"],
        df.loc[df["capacity_event"] == 1, "weather_severity"],
    ],
    bins=20,
    label=["Normal", "Capacity event"],
    alpha=0.7,
)

plt.xlabel("Weather severity")
plt.ylabel("Number of observations")
plt.legend()
plt.show()

## 4. Separación entre features y target

Recordatorio:

- `X`: variables que utiliza el modelo.
- `y`: valor que queremos predecir.

In [ ]:
X = df.drop(columns="capacity_event")
y = df["capacity_event"]

X.head()

## 5. Separación TRAIN / TEST

Reservamos el **20 % como test final**.

El test no se utilizará:

- para entrenar;
- para elegir modelo;
- para seleccionar hiperparámetros.

Todo el desarrollo se hará únicamente sobre `X_train` / `y_train`.

Al final abriremos el test una sola vez.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape, y_train.shape)
print("Test: ", X_test.shape, y_test.shape)
print()
print("Positive rate train:", round(float(y_train.mean()), 3))
print("Positive rate test: ", round(float(y_test.mean()), 3))

## 6. Preprocesamiento

Tenemos dos tipos de variables:

### Numéricas

Usaremos `StandardScaler`.

### Categóricas

Usaremos `OneHotEncoder`.

`ColumnTransformer` permite aplicar automáticamente el tratamiento apropiado a cada grupo de columnas.

El `Pipeline` garantizará además que el preprocesamiento se ajusta **solo con los datos de entrenamiento de cada fold**, evitando *data leakage* durante cross-validation.

In [ ]:
numeric_features = [
    "traffic_count",
    "sector_capacity",
    "weather_severity",
    "avg_delay_min",
    "staffing_ratio",
    "coordination_load",
]

categorical_features = [
    "sector_type",
    "time_band",
    "flow_state",
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ]
)

preprocessor

## 7. Baseline

Antes de intentar construir un modelo inteligente necesitamos responder:

> ¿Es nuestro modelo mejor que una estrategia trivial?

Usaremos `DummyClassifier`, que será nuestra referencia mínima.

Un modelo complejo que no supera claramente al baseline no aporta valor.

In [ ]:
baseline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            DummyClassifier(strategy="most_frequent"),
        ),
    ]
)

baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_train)

print(
    "Baseline train accuracy:",
    round(accuracy_score(y_train, baseline_predictions), 3)
)

## 8. Modelos candidatos

Compararemos tres modelos que conectan directamente con lo estudiado:

1. **Logistic Regression**  
   Modelo lineal de clasificación.

2. **Decision Tree**  
   Modelo capaz de aprender relaciones no lineales, pero susceptible a overfitting.

3. **Neural Network (MLP)**  
   Una pequeña red neuronal con una capa oculta.

No usaremos todavía modelos más avanzados. El objetivo es comprobar que dominamos correctamente el proceso, no maximizar la puntuación.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE,
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=15,
        random_state=RANDOM_STATE,
    ),
    "Neural Network": MLPClassifier(
        hidden_layer_sizes=(20,),
        max_iter=1200,
        early_stopping=True,
        random_state=RANDOM_STATE,
    ),
}

## 9. Cross-validation para comparar modelos

Utilizaremos 5 folds.

Mediremos:

- `accuracy`;
- `precision`;
- `recall`;
- `f1`.

En este caso ATM consideraremos especialmente importante el **recall de la clase positiva**, porque un falso negativo representa una situación de alta carga que el modelo no ha detectado.

Esto no significa que recall sea siempre la métrica correcta: la métrica debe derivarse del coste operacional de los errores.

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}

rows = []

for model_name, classifier in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", classifier),
        ]
    )

    cv_result = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring=scoring,
        n_jobs=-1,
    )

    rows.append({
        "model": model_name,
        "accuracy": cv_result["test_accuracy"].mean(),
        "precision": cv_result["test_precision"].mean(),
        "recall": cv_result["test_recall"].mean(),
        "f1": cv_result["test_f1"].mean(),
        "f1_std": cv_result["test_f1"].std(),
    })

cv_results = (
    pd.DataFrame(rows)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

cv_results.round(3)

In [ ]:
cv_results.set_index("model")[["precision", "recall", "f1"]].plot(
    kind="bar",
    ylim=(0, 1)
)

plt.ylabel("Cross-validation score")
plt.xlabel("")
plt.xticks(rotation=20)
plt.show()

## 10. Hyperparameter tuning del Decision Tree

Ahora repetimos correctamente el experimento que utilizamos para estudiar overfitting.

No miraremos el test.

`GridSearchCV`:

1. prueba distintos hiperparámetros;
2. evalúa cada combinación mediante cross-validation;
3. selecciona la mejor según la métrica elegida.

Usaremos `f1` como criterio de selección para mantener un equilibrio entre precision y recall.

In [ ]:
tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=RANDOM_STATE
            ),
        ),
    ]
)

param_grid = {
    "classifier__max_depth": [3, 4, 5, 6, 8, 10, None],
    "classifier__min_samples_leaf": [5, 10, 20, 40],
}

grid = GridSearchCV(
    estimator=tree_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    return_train_score=True,
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)
print()
print("Best CV F1:", round(grid.best_score_, 3))

### ¿Dónde aparece el overfitting?

Podemos comparar el score medio de entrenamiento con el de validación.

Cuando un modelo se vuelve demasiado complejo:

- el rendimiento en train puede seguir creciendo;
- el rendimiento en validación deja de mejorar o empeora.

Ese gap es una señal de overfitting.

In [ ]:
grid_results = pd.DataFrame(grid.cv_results_)

tree_summary = grid_results[
    [
        "param_classifier__max_depth",
        "param_classifier__min_samples_leaf",
        "mean_train_score",
        "mean_test_score",
    ]
].copy()

tree_summary["generalization_gap"] = (
    tree_summary["mean_train_score"]
    - tree_summary["mean_test_score"]
)

tree_summary.sort_values(
    "mean_test_score",
    ascending=False
).head(10).round(3)

## 11. Selección del modelo final

Elegiremos entre:

- Logistic Regression;
- Neural Network;
- Decision Tree optimizado.

Seguimos sin usar el test.

Para disponer de métricas comparables, calculamos cross-validation de los tres candidatos finales.

In [ ]:
final_candidates = {
    "Logistic Regression": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "Tuned Decision Tree": grid.best_estimator_,
    "Neural Network": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                MLPClassifier(
                    hidden_layer_sizes=(20,),
                    max_iter=1200,
                    early_stopping=True,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}

rows = []

for model_name, model in final_candidates.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=5,
        scoring=scoring,
        n_jobs=-1,
    )

    rows.append({
        "model": model_name,
        "accuracy": scores["test_accuracy"].mean(),
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean(),
    })

final_cv_results = (
    pd.DataFrame(rows)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

final_cv_results.round(3)

In [ ]:
best_model_name = final_cv_results.iloc[0]["model"]
best_model = final_candidates[best_model_name]

print("Selected model:", best_model_name)

## 12. Entrenamiento definitivo

Hasta ahora hemos utilizado cross-validation sobre el conjunto de entrenamiento.

Ahora entrenamos el modelo seleccionado utilizando **todo el training set**.

In [ ]:
best_model.fit(X_train, y_train)

# 13. TEST FINAL

Ahora, y solo ahora, utilizamos `X_test`.

Esta es nuestra estimación final del rendimiento sobre datos que no han intervenido en ninguna decisión durante el desarrollo.

In [ ]:
test_predictions = best_model.predict(X_test)

test_metrics = pd.Series({
    "accuracy": accuracy_score(y_test, test_predictions),
    "precision": precision_score(y_test, test_predictions),
    "recall": recall_score(y_test, test_predictions),
    "f1": f1_score(y_test, test_predictions),
})

test_metrics.round(3)

In [ ]:
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=["Normal", "Capacity event"],
    )
)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    display_labels=["Normal", "Capacity event"],
)

plt.show()

## 14. Interpretación operacional de la matriz de confusión

Para este problema:

- **True Positive (TP)**: había evento de capacidad y lo detectamos.
- **True Negative (TN)**: situación normal correctamente identificada.
- **False Positive (FP)**: generamos una alerta que no era necesaria.
- **False Negative (FN)**: existía un evento de capacidad y no lo detectamos.

En un sistema ATM real, el coste de FP y FN debería definirse junto a expertos operacionales.

Esto determina qué métrica deberíamos optimizar y qué threshold de decisión sería aceptable.

Por tanto:

> **una métrica de ML no tiene significado operacional por sí sola.**

## 15. Probabilidades y threshold

Un clasificador no tiene por qué limitarse a producir 0 o 1.

Si dispone de `predict_proba`, podemos obtener una estimación de probabilidad.

In [ ]:
if hasattr(best_model.named_steps["classifier"], "predict_proba"):
    test_probabilities = best_model.predict_proba(X_test)[:, 1]

    probability_preview = pd.DataFrame({
        "real": y_test.to_numpy()[:10],
        "probability_capacity_event": test_probabilities[:10],
        "prediction_default_threshold": test_predictions[:10],
    })

    display(probability_preview.round(3))
else:
    print("El modelo seleccionado no proporciona predict_proba.")

El threshold habitual es aproximadamente `0.5`.

Pero podríamos reducirlo, por ejemplo a `0.35`, para favorecer recall y detectar más eventos a costa de aumentar los falsos positivos.

En un proyecto real el threshold sería una **decisión operacional**, no simplemente un valor matemático por defecto.

In [ ]:
if hasattr(best_model.named_steps["classifier"], "predict_proba"):
    threshold = 0.35

    lower_threshold_predictions = (
        test_probabilities >= threshold
    ).astype(int)

    threshold_comparison = pd.DataFrame({
        "metric": ["precision", "recall", "f1"],
        "threshold_0.50": [
            precision_score(y_test, test_predictions),
            recall_score(y_test, test_predictions),
            f1_score(y_test, test_predictions),
        ],
        "threshold_0.35": [
            precision_score(y_test, lower_threshold_predictions),
            recall_score(y_test, lower_threshold_predictions),
            f1_score(y_test, lower_threshold_predictions),
        ],
    })

    display(threshold_comparison.round(3))

## 16. Interpretación básica de features

La forma exacta de interpretar el modelo depende del algoritmo seleccionado.

Para mantener el ejercicio sencillo, entrenaremos además una Logistic Regression y observaremos sus coeficientes.

**Importante:** esto describe asociación aprendida por el modelo, no causalidad.

In [ ]:
interpret_model = final_candidates["Logistic Regression"]
interpret_model.fit(X_train, y_train)

feature_names = (
    interpret_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    interpret_model
    .named_steps["classifier"]
    .coef_[0]
)

feature_effects = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
})

feature_effects["abs_coefficient"] = (
    feature_effects["coefficient"].abs()
)

feature_effects.sort_values(
    "abs_coefficient",
    ascending=False
).head(15)

Interpretación simplificada:

- coeficiente positivo → aumenta la probabilidad estimada del evento;
- coeficiente negativo → la reduce;
- mayor magnitud absoluta → mayor influencia en la frontera lineal.

Como las variables numéricas están estandarizadas, sus coeficientes resultan más comparables entre sí que si utilizásemos escalas originales muy diferentes.

## 17. Inferencia sobre una nueva situación ATM

Finalmente utilizamos el modelo como se usaría en producción: recibimos un caso nuevo y generamos una predicción.

Este registro **no forma parte del dataset de entrenamiento**.

In [ ]:
new_situation = pd.DataFrame([{
    "traffic_count": 43,
    "sector_capacity": 36,
    "weather_severity": 7.5,
    "avg_delay_min": 18.0,
    "staffing_ratio": 0.86,
    "coordination_load": 8.0,
    "sector_type": "TMA",
    "time_band": "afternoon",
    "flow_state": "growing",
}])

prediction = best_model.predict(new_situation)[0]

print(
    "Predicted class:",
    "CAPACITY EVENT" if prediction == 1 else "NORMAL"
)

if hasattr(best_model.named_steps["classifier"], "predict_proba"):
    probability = best_model.predict_proba(new_situation)[0, 1]
    print(
        "Estimated probability:",
        round(float(probability), 3)
    )

# 18. Qué hemos hecho de principio a fin

El flujo completo ha sido:

```text
PROBLEMA OPERACIONAL
        ↓
DEFINIR TARGET
        ↓
DATOS
        ↓
EDA / CALIDAD
        ↓
X / y
        ↓
TRAIN ───────────── TEST
  │                     │
  ↓                     │
PREPROCESSING           │
  │                     │
  ↓                     │
BASELINE                │
  │                     │
  ↓                     │
MODELOS                 │
  │                     │
  ↓                     │
CROSS-VALIDATION        │
  │                     │
  ↓                     │
HIPERPARÁMETROS         │
  │                     │
  ↓                     │
SELECCIÓN MODELO        │
  │                     │
  ↓                     │
ENTRENAMIENTO FINAL     │
  │                     │
  └──────────────────→ TEST FINAL
                        │
                        ↓
                MÉTRICAS + ERRORES
                        │
                        ↓
                DECISIÓN OPERACIONAL
```

# 19. Preguntas de comprobación

Antes de dar por terminado el notebook, deberías poder responder sin consultar documentación:

1. ¿Cuál es la diferencia entre `X` e `y`?
2. ¿Por qué no debemos utilizar el test para elegir hiperparámetros?
3. ¿Qué diferencia hay entre parámetros e hiperparámetros?
4. ¿Qué problema evita el `Pipeline` durante cross-validation?
5. ¿Qué es overfitting?
6. ¿Por qué un accuracy alto puede ser engañoso?
7. ¿Qué diferencia existe entre precision y recall?
8. ¿Por qué necesitamos un baseline?
9. ¿Por qué un modelo más complejo no es necesariamente mejor?
10. ¿Qué significa que el modelo generalice?
11. ¿Por qué el threshold puede convertirse en una decisión operacional?
12. ¿Por qué una correlación o un coeficiente del modelo no demuestra causalidad?

Si puedes explicar estas cuestiones con tus propias palabras, el objetivo del bloque está cumplido.

# 20. Cierre del bloque: ML fundamental

Con este proyecto damos por cubierto el **núcleo mínimo de Machine Learning** necesario para continuar hacia IA aplicada:

- regresión lineal;
- regresión logística;
- clasificación;
- datos numéricos y categóricos;
- preprocessing;
- train / validation / test;
- cross-validation;
- overfitting;
- parámetros e hiperparámetros;
- métricas;
- árboles;
- redes neuronales;
- pipelines;
- evaluación end-to-end.

No significa que Machine Learning esté agotado como disciplina. Quedan muchos temas avanzados: ensembles, boosting, feature engineering, calibración, series temporales, interpretabilidad avanzada, MLOps, etc.

Pero **no necesitamos estudiarlos ahora para avanzar hacia nuestro objetivo**.

---

# Próximo bloque — Embeddings y LLM aplicado

A partir de aquí cambia el foco:

```text
ML FUNDAMENTAL
      ✓
      │
      ▼
EMBEDDINGS
      │
      ▼
BÚSQUEDA SEMÁNTICA
      │
      ▼
TOKENS + TRANSFORMERS
      │
      ▼
LLM
      │
      ├──────── RAG
      │
      ├──────── TOOL CALLING
      │
      └──────── AGENTES
```

El siguiente trabajo práctico ya no consistirá en entrenar otro clasificador clásico.

Construiremos una pequeña **búsqueda semántica ATM mediante embeddings** y la utilizaremos como puente hacia un primer sistema RAG.